# LLM classification - authorship verification by prompting

Asks an LLM (MODEL_ID) whether the two texts of every pair were written by the same author.
The prompt is a repository file, the model decodes greedily in vLLM and only the "answer" value of its reply counts.

- test - every PAN 2021 test pair
- pilot - 256 PAN 2020 training pairs, to set the answer budget before any test pair runs
- smoke - the first 12 pilot pairs, to check the run on the GPU

A restarted run continues where it stopped - the answers are saved in chunks, a finished run is not repeated and chunks of other settings stop the run.

In [ ]:
MODEL_ID = ""
MODE = "test"  # "test", "pilot" or "smoke"
SEED = 42  # seed of the engine
RUN_TAG = ""  # label in the file names of a repeated run
MAX_TOKENS = None

Imports - vLLM schedules deterministically in one process, set before it is imported

In [ ]:
import logging
import os
import re
from dataclasses import asdict
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
import vllm
from vllm import LLM, SamplingParams

from avcore import data, evaluation, llm, progress, runs, tracking

The models - everything in which the runs of the three models differ

Qwen and Gemma run with a 32 768-token context, so the A40 memory holds several long requests at once, and each text is cut to its first 14 000 tokens, so two texts, the prompt and the answer always fit (with texts of about 21 000 characters, the cut touches only rare outliers).
Llama-2 can read at most 4 096 tokens (a limit of the model itself, not of the GPU or vLLM), so each text is cut to 1 750 tokens and the answer gets whatever room is left; max_tokens of Qwen and Gemma stays None until the pilot sets it.

In [ ]:
MODELS = {
    "Qwen/Qwen3-4B-Instruct-2507": {
        "name": "Qwen3-4B-Instruct-2507",
        "revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
        "chat_template_file": None,
        "text_token_limit": 14_000,
        "quantization": None,
        "dtype": "auto",
        "max_model_len": 32_768,
        "max_tokens": None,
    },
    "google/gemma-3-4b-it": {
        "name": "gemma-3-4b-it",
        "revision": "093f9f388b31de276ce2de164bdc2081324b9767",
        "chat_template_file": None,
        "text_token_limit": 14_000,
        "quantization": None,
        "dtype": "auto",
        "max_model_len": 32_768,
        "max_tokens": None,
    },
    "TheBloke/Llama-2-70B-chat-GPTQ": {
        "name": "Llama-2-70B-chat-GPTQ",
        "revision": "ac53ed5e8ebdbbe6db22ecf816e21f55905418df",
        # the GPTQ tokenizer has no chat template - the file holds the one of Llama-2-chat
        "chat_template_file": "llama2_chat_template.jinja",
        "text_token_limit": 1_750,
        "quantization": "gptq_marlin",
        "dtype": "float16",
        "max_model_len": 4_096,
        # no budget of its own - only the 4 096-token context caps the answer
        "max_tokens": 4_096,
    },
}
MODES = ("test", "pilot", "smoke")

Check the parameters - before anything is loaded

In [ ]:
if MODEL_ID not in MODELS:
    raise ValueError(f"MODEL_ID {MODEL_ID!r} is not one of {list(MODELS)}.")
if MODE not in MODES:
    raise ValueError(f"MODE {MODE!r} is not one of {MODES}.")
if MAX_TOKENS is not None and (type(MAX_TOKENS) is not int or MAX_TOKENS < 1):
    raise ValueError(f"MAX_TOKENS {MAX_TOKENS!r} is not a positive whole number.")
if MODE == "test" and MAX_TOKENS is not None:
    raise ValueError(
        "A test run uses the max_tokens of the model table, not MAX_TOKENS.",
    )
MODEL = MODELS[MODEL_ID]
max_tokens = MODEL["max_tokens"] if MAX_TOKENS is None else MAX_TOKENS
if max_tokens is None:
    raise ValueError(f"max_tokens of {MODEL['name']} is not set - run the pilot first.")

Constants - the prompt, decoding and the rule for unanswered pairs

In [ ]:
PROMPT_FILE = "prompt_llama2_v1.json"
CLASSIFICATION_TYPE = "prompt"
DEVELOPER_INITIALS = "JP"
RUN_IDENTITY = {
    "family": "llm",
    "model": MODEL["name"],
    "representation": "none",
    "combination": "none",
    "classifier": CLASSIFICATION_TYPE,
}
DECODING = llm.DecodingSettings(
    temperature=0.0,
    top_p=1.0,
    max_tokens=max_tokens,
    seed=SEED,
)
# a reply without a valid answer counts as "different author", and its score 0,5 (instead of
# 0 or 1) marks it as unanswered in the result tables
UNANSWERED_PRED = llm.DIFFERENT_AUTHOR
UNANSWERED_SCORE = 0.5
# True replaces the tokenizer's control strings (such as "</s>") found inside a text by a space;
# False leaves them to the cut, which deletes them and joins the words around them; set to True
# if any training text contains one
REPLACE_SPECIAL_TOKENS = False
if REPLACE_SPECIAL_TOKENS is None:
    raise ValueError("REPLACE_SPECIAL_TOKENS is not set - run the markup count first.")
SMOKE_PAIRS = 12
CHUNK_SIZE = 4 if MODE == "smoke" else 256
ENGINE_SETTINGS = {
    "model": MODEL_ID,
    "revision": MODEL["revision"],
    "dtype": MODEL["dtype"],
    "quantization": MODEL["quantization"],
    "max_model_len": MODEL["max_model_len"],
    "seed": SEED,
    # off, so every prompt is computed from scratch; with it on, a prompt reuses parts computed for
    # earlier prompts, and a restarted run (empty cache) could give slightly different answers
    "enable_prefix_caching": False,
    # vLLM may use 90 % of the GPU memory; the memory left after the model weights decides how many
    # prompts run at once, which can change the answers slightly, so the value is fixed here and
    # saved with the run
    "gpu_memory_utilization": 0.9,
}

Paths and the run name - the pilot pairs are made by a check script, outside the repository

In [ ]:
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
PROMPT_DIR = PROJECT_DIR / "prompts"
if MODE == "test":
    PAIRS_PATH = DATA_DIR / "raw" / data.TEST_PAIRS_FILE
    TRUTH_PATH = DATA_DIR / "raw" / data.TEST_TRUTH_FILE
else:
    PAIRS_PATH = DATA_DIR / "processed" / "pan20-llm-pilot-pairs.jsonl"
    TRUTH_PATH = DATA_DIR / "processed" / "pan20-llm-pilot-truth.jsonl"
PROMPT_PATH = PROMPT_DIR / PROMPT_FILE
TEMPLATE_FILE = MODEL["chat_template_file"]
TEMPLATE_PATH = None if TEMPLATE_FILE is None else PROMPT_DIR / TEMPLATE_FILE

SPLIT = "test" if MODE == "test" else "pilot"
name_parts = [DEVELOPER_INITIALS, MODEL["name"], CLASSIFICATION_TYPE]
# a pilot or smoke run carries its mode in the name, so it never overwrites the test run
if MODE != "test":
    name_parts.append(MODE)
if RUN_TAG:
    name_parts.append(RUN_TAG)
name_parts.append("v2")
RUN_NAME = "_".join(name_parts)
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(f"RUN_NAME {RUN_NAME!r} is not a plain file name.")
RESULTS_DIRS = {
    "test": Path.home() / "results_v2",
    "pilot": Path.home() / "results_v2_pilot",
    "smoke": Path.home() / "results_v2_smoke" / RUN_TAG,
}
RUN_DIR = RESULTS_DIRS[MODE] / RUN_NAME
ANSWERS_DIR = Path.home() / "llm_answers_v2" / RUN_NAME
PROGRESS_PATH = Path.home() / "logs" / f"{RUN_NAME}.progress.json"

Logging, the progress file and the run context - commit, lock and data hashes, versions

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s - %(message)s",
)
progress_file = progress.ProgressFile(PROGRESS_PATH, RUN_NAME)
progress_file.start_stage("start")
data_paths = [PAIRS_PATH, TRUTH_PATH, PROMPT_PATH]
if TEMPLATE_PATH is not None:
    data_paths.append(TEMPLATE_PATH)
run_context = tracking.collect_run_context(PROJECT_DIR, data_paths)
print(
    f"Commit {run_context.environment['commit']}, GPU {run_context.environment['gpu_name']}",
)

Load the pairs - the raw texts; the LLMs read no cleaned file

In [ ]:
pairs_df = data.load_pan_pairs(PAIRS_PATH, TRUTH_PATH, clean_texts=False)
if not pairs_df["id"].is_unique:
    raise ValueError(
        "The pairs file repeats an id - the prediction table needs unique ids.",
    )
if MODE == "smoke":
    pairs_df = pairs_df.head(SMOKE_PAIRS)
print(f"{len(pairs_df)} {SPLIT} pairs")

Load the tokenizer of the model at its pinned revision

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL["revision"])
# the special tokens of this tokenizer that are text (<|im_end|>, <end_of_turn>, </s>, ...)
special_strings = []
for token in tokenizer.all_special_tokens:
    # an empty or whitespace-only token would match almost everywhere in a text
    if token.strip():
        special_strings.append(token)
print(f"{len(special_strings)} special token strings")

Replace markup strings in the texts by a space - a template string (such as [INST] or ```) could close the text delimiter or open a chat turn, so it is always replaced; a special token (such as </s>) is counted in every run and replaced only if REPLACE_SPECIAL_TOKENS is True

In [ ]:
cleaned_pairs = []
n_template_strings_replaced = 0
n_special_tokens_found = 0
for pair in pairs_df["pair"]:
    cleaned_pair = []
    for text in pair:
        # the template strings are always replaced
        text, n_found = llm.neutralize_template_strings(text)
        n_template_strings_replaced += n_found

        # the special tokens are always counted, but replaced only if the switch is on
        text_without_special, n_found = llm.neutralize_template_strings(
            text,
            special_strings,
        )
        n_special_tokens_found += n_found
        if REPLACE_SPECIAL_TOKENS:
            text = text_without_special

        cleaned_pair.append(text)
    cleaned_pairs.append(cleaned_pair)
pairs_df = pairs_df.assign(pair=cleaned_pairs)
print(f"Template strings replaced: {n_template_strings_replaced}")
print(
    f"Special tokens found: {n_special_tokens_found} (replaced: {REPLACE_SPECIAL_TOKENS})",
)

Count the texts longer than the token limit of the model - only their first tokens reach the prompt

In [ ]:
text_lengths = []
for pair in pairs_df["pair"]:
    for text in pair:
        token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        text_lengths.append(len(token_ids))
n_texts_cut = 0
for length in text_lengths:
    if length > MODEL["text_token_limit"]:
        n_texts_cut += 1
print(
    f"{n_texts_cut} of {len(text_lengths)} texts cut to {MODEL['text_token_limit']} tokens",
)

Build the prompts - the run stops here if a prompt fills the whole context, because the engine would refuse it and with it the whole chunk

In [ ]:
# None keeps the chat template of the tokenizer
chat_template = None
if TEMPLATE_PATH is not None:
    chat_template = TEMPLATE_PATH.read_text(encoding="utf-8")
builder = llm.PromptBuilder(
    tokenizer=tokenizer,
    prompt=llm.load_prompt(PROMPT_PATH),
    text_token_limit=MODEL["text_token_limit"],
    chat_template=chat_template,
)
prompt_tokens = llm.count_prompt_tokens(builder, pairs_df)
llm.check_prompt_lengths(prompt_tokens, MODEL["max_model_len"])
print(f"Prompt tokens - min {prompt_tokens.min()}, max {prompt_tokens.max()}")

Settings of the run - a finished run with the same name is skipped, but only if it holds the same settings; with other settings the notebook stops

In [ ]:
run_params = dict(RUN_IDENTITY)

# the model
run_params["model_id"] = MODEL_ID
run_params["model_revision"] = MODEL["revision"]
run_params["version.vllm"] = vllm.__version__

# the code and the data - a run folder of another commit or other data is never taken as finished
pairs_sha256 = run_context.environment[f"data_sha256.{PAIRS_PATH.name}"]
truth_sha256 = run_context.environment[f"data_sha256.{TRUTH_PATH.name}"]
run_params["code_commit"] = run_context.environment["commit"]
run_params["pairs_file_sha256"] = pairs_sha256
run_params["truth_file_sha256"] = truth_sha256

# the run
run_params["mode"] = MODE
run_params["seed"] = SEED
run_params["chunk_size"] = CHUNK_SIZE
run_params["pairs"] = len(pairs_df)
if MODE == "test":
    run_params["mlflow_experiment"] = tracking.EXPERIMENT_NAME
else:
    run_params["mlflow_experiment"] = tracking.SMOKE_EXPERIMENT_NAME

# the prompt and the texts in it
run_params["prompt_file"] = PROMPT_FILE
run_params["prompt_sha256"] = builder.prompt.sha256
if TEMPLATE_FILE is None:
    run_params["chat_template_file"] = "tokenizer"
else:
    run_params["chat_template_file"] = TEMPLATE_FILE
run_params["text_token_limit"] = MODEL["text_token_limit"]
run_params["template_strings"] = " ".join(llm.TEMPLATE_STRINGS)
run_params["replace_special_tokens"] = REPLACE_SPECIAL_TOKENS

# the rule for unanswered pairs
run_params["unanswered_pred"] = UNANSWERED_PRED
run_params["unanswered_score"] = UNANSWERED_SCORE

# the decoding and the engine
for name, value in asdict(DECODING).items():
    run_params[f"decoding.{name}"] = value
for name, value in ENGINE_SETTINGS.items():
    # the model is already in the params as model_id
    if name != "model":
        run_params[f"engine.{name}"] = value

finished = runs.find_finished_run(RUN_DIR, run_params)
if finished:
    print(f"{RUN_DIR} exists - the run is already generated and evaluated.")

Generate the answers - a chunk that already has an answer file is skipped, and the engine is not loaded if every chunk has one

In [ ]:
def report_progress(
    chunks_done: int,
    n_chunks: int,
    progress_file: progress.ProgressFile,
    chunk_size: int,
    n_pairs: int,
) -> None:
    """Write the number of answered pairs to the progress file after each saved chunk.

    Args:
        chunks_done: Chunks with a saved answer file, those of earlier runs included.
        n_chunks: Chunks of the whole run (given by generate_answers, not needed here).
        progress_file: Progress file of the run.
        chunk_size: Pairs per chunk.
        n_pairs: Pairs of the whole run.
    """
    # the last chunk can be shorter, so the count is capped at the number of pairs
    pairs_done = min(chunks_done * chunk_size, n_pairs)
    progress_file.update(force=True, pairs_done=pairs_done, n_pairs=n_pairs)


if not finished:
    pair_ids = pairs_df["id"].tolist()
    # the cleaned texts of these files, read by this code, are what the engine reads
    description_settings = dict(ENGINE_SETTINGS)
    description_settings.update(
        {
            "pairs_sha256": pairs_sha256,
            "template_strings": list(llm.TEMPLATE_STRINGS),
            "replace_special_tokens": REPLACE_SPECIAL_TOKENS,
            "tokenizer_revision": MODEL["revision"],
            "code_commit": run_context.environment["commit"],
        },
    )
    run_description = llm.describe_run(
        engine_settings=description_settings,
        builder=builder,
        decoding=DECODING,
        chunk_size=CHUNK_SIZE,
        pair_ids=pair_ids,
    )
    llm.prepare_answers_dir(ANSWERS_DIR, run_description)
    n_chunks = len(llm.split_chunks(pair_ids, CHUNK_SIZE))
    n_chunks_done = len(llm.find_finished_chunks(ANSWERS_DIR))
    if n_chunks_done < n_chunks:
        n_pairs = len(pair_ids)
        progress_file.start_stage("generation", pairs_done=0, n_pairs=n_pairs)
        engine = LLM(**ENGINE_SETTINGS)
        llm.generate_answers(
            engine=engine,
            sampling_params=SamplingParams(**asdict(DECODING)),
            builder=builder,
            pairs_df=pairs_df,
            answers_dir=ANSWERS_DIR,
            chunk_size=CHUNK_SIZE,
            # the progress watcher draws its bar from pairs_done and n_pairs
            on_chunk=partial(
                report_progress,
                progress_file=progress_file,
                chunk_size=CHUNK_SIZE,
                n_pairs=n_pairs,
            ),
        )
        del engine

Parse the answers and evaluate - only here are the answers compared with the true labels, the prompts never contain them

In [ ]:
if not finished:
    progress_file.start_stage("evaluation")
    generated = llm.read_answers(ANSWERS_DIR, pairs_df["id"].tolist(), CHUNK_SIZE)
    tables = llm.build_prediction_tables(
        pairs_df=pairs_df,
        answers_df=generated.answers_df,
        unanswered_pred=UNANSWERED_PRED,
        unanswered_score=UNANSWERED_SCORE,
    )
    predictions_df = tables.predictions_df
    split_metrics = evaluation.evaluate_results(
        predictions_df["same"],
        predictions_df["pred"],
    )
    print(f"{SPLIT} {split_metrics}, unanswered pairs {len(tables.unanswered_df)}")

Confusion matrix - rows are the true labels, columns the predictions (True = same author)

In [ ]:
if not finished:
    matrix = confusion_matrix(
        predictions_df["same"],
        predictions_df["pred"],
        labels=[0, 1],
    )
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[False, True],
        yticklabels=[False, True],
        ax=axis,
        cbar=False,
    )
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title(f"Confusion Matrix: {MODEL['name']}")

Metrics of the run - the answer counts by status and by finish reason

In [ ]:
if not finished:
    generation_df = tables.generation_df
    statuses = generation_df["status"]
    finish_reasons = generation_df["finish_reason"]

    # the evaluation metrics of the split, such as test_f1
    metrics = {}
    for name, value in split_metrics.to_dict().items():
        metrics[f"{SPLIT}_{name}"] = value

    # the answers by status
    metrics["n_unanswered"] = len(tables.unanswered_df)
    metrics["n_no_answer"] = int((statuses == llm.NO_ANSWER).sum())
    metrics["n_conflicting_answers"] = int((statuses == llm.CONFLICTING_ANSWERS).sum())

    # the answers by finish reason - "stop": the model ended its answer, "length": it was cut
    is_cut = finish_reasons == "length"
    metrics["n_finish_stop"] = int((finish_reasons == "stop").sum())
    metrics["n_finish_length"] = int(is_cut.sum())
    metrics["n_finish_other"] = int((~finish_reasons.isin(["stop", "length"])).sum())

    # a cut answer hit the context if prompt and answer fill max_model_len, otherwise max_tokens
    used_tokens = generation_df["prompt_tokens"] + generation_df["generated_tokens"]
    is_cut_by_context = is_cut & (used_tokens >= MODEL["max_model_len"])
    is_cut_answered = is_cut & (statuses == llm.ANSWERED)
    metrics["n_cut_by_context"] = int(is_cut_by_context.sum())
    metrics["n_cut_by_budget"] = int((is_cut & ~is_cut_by_context).sum())
    metrics["n_cut_answered"] = int(is_cut_answered.sum())
    metrics["n_cut_unanswered"] = int((is_cut & ~is_cut_answered).sum())

    # the texts and the prompts - 500 is the smallest answer budget of the pilot rule
    answer_room = MODEL["max_model_len"] - prompt_tokens
    metrics["n_texts_cut"] = n_texts_cut
    metrics["n_special_tokens_found"] = n_special_tokens_found
    metrics["n_template_strings_replaced"] = n_template_strings_replaced
    metrics["max_prompt_tokens"] = int(prompt_tokens.max())
    metrics["min_answer_room"] = int(answer_room.min())
    metrics["n_prompts_room_below_500"] = int((answer_room < 500).sum())

    metrics["generation_seconds"] = generated.generation_seconds

Write the run folder - the tables hold numbers and short codes, never a text; a pilot or smoke run names its table answers-pilot, because its pairs are training pairs, not test predictions

In [ ]:
if not finished:
    if MODE == "test":
        label_table = "predictions-test"
    else:
        label_table = "answers-pilot"
    runs.remove_unfinished_run_dir(RUN_DIR)
    tracking.write_run_files(
        RUN_DIR,
        context=run_context,
        params=run_params,
        metrics=metrics,
        tables={
            label_table: predictions_df,
            f"unanswered-{SPLIT}": tables.unanswered_df,
            f"generation-{SPLIT}": generation_df,
        },
        figures={f"confusion_matrix-{SPLIT}": figure},
    )
progress_file.start_stage("done")

The run folder and its metrics

In [ ]:
print(f"Run folder {RUN_DIR}")
print((RUN_DIR / tracking.METRICS_FILE).read_text(encoding="utf-8"))